**# RAGify PDF Assistant**

AI-powered PDF Question Answering System built using:

- Python
- LangChain
- HuggingFace Embeddings
- FAISS Vector Database
- Retrieval-Augmented Generation **(RAG)**
- TinyLlama LLM

Workflow:

PDF → Chunking → Embeddings → FAISS → Retriever → LLM → Answer

**## Project Workflow**

1. Upload PDF
2. Extract Text from PDF
3. Split Text into Chunks
4. Generate Embeddings
5. Store Embeddings in FAISS
6. Retrieve Relevant Chunks
7. Send Context to LLM
8. Generate Final Answer

In [1]:
!pip install -q -U langchain langchain-community langchain-text-splitters langchain-google-genai pypdf faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.3/346.3 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
!pip install -q sentence-transformers

In [3]:
!pip install -q transformers accelerate torch

In [4]:
!pip install -q sentencepiece

In [6]:
import langchain
print("LangChain Installed Successfully")

LangChain Installed Successfully


In [5]:
from google.colab import files
uploaded = files.upload()

Saving pdf1.pdf to pdf1.pdf


In [7]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader('/content/pdf1.pdf')
document = loader.load()
print("Total Pages:", len(document))

/tmp/ipykernel_3243/3619694317.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Total Pages: 15


In [8]:
print(document[0].page_content[:1000])

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Exp

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunk = text_splitter.split_documents(document)
print('total Chunks:',len(chunk))

total Chunks: 52


In [10]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("HuggingFace Embeddings Loaded Successfully")

/tmp/ipykernel_3243/1101236968.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

HuggingFace Embeddings Loaded Successfully


In [11]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(
    chunk,
    embeddings
)

print("FAISS Vector Database Created Successfully")

FAISS Vector Database Created Successfully


In [12]:
retriever = vector_store.as_retriever(search_kwargs={'k':3})
print("Retriever Created Successfully")

Retriever Created Successfully


In [15]:
import transformers

print(transformers.__version__)

5.9.0


In [16]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0"
)

print("TinyLlama Loaded Successfully")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

TinyLlama Loaded Successfully


In [17]:
query = input("Ask a question: ")

docs = retriever.invoke(query)

context = "\n\n".join([doc.page_content for doc in docs])

prompt = f"""
You are a helpful AI assistant.

Answer ONLY from the provided context.
If the answer is not in the context, say:
"I could not find the answer in the document."

Context:
{context}

Question:
{query}

Answer:
"""

result = generator(
    prompt,
    max_new_tokens=150,
    return_full_text=False
)

print(result[0]["generated_text"])

Ask a question: self attention


[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


self attention is a recurrent neural network (RNN) layer in transformer models.


   Technologies Used
- Python
- Google Colab
- LangChain
- HuggingFace Embeddings
- FAISS
- TinyLlama
- Retrieval-Augmented Generation (RAG)

 Features

- Upload PDF Documents
- Extract PDF Text
- Semantic Search using FAISS
- Context Retrieval
- AI-powered Question Answering
- Retrieval-Augmented Generation (RAG)